# Importando as bibliotecas necessárias

In [256]:
import os
from requests import post, get
import json
from base64 import b64encode
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import numpy as np

# Definindo as variáveis iniciais

In [429]:
client_id = os.environ['CLIENT_ID_SPOTIFY']
client_secret = os.environ['CLIENT_SECRET_SPOTIFY']
base_url = 'https://accounts.spotify.com'
endpoint = '/api/token'

# Criando as funções para acesso

In [432]:
def request_auth():
    b64_encoded = b64encode(f'{client_id}:{client_secret}'.encode())
    auth_b64 = str(b64_encoded, 'utf-8')
    
    auth_options = {
        'url': base_url + endpoint,
        'headers':{
            'Authorization': 'Basic ' + auth_b64,
            'Content-Type': 'application/x-www-form-urlencoded'
        },
        'form':{
            'grant_type': 'client_credentials'
        },
        'json': True
    }

    request = post(url=auth_options['url'], headers=auth_options['headers'], data=auth_options['form'])
    if request.status_code == 200:
        json_request = request.json()
        access_token = json_request['access_token']
        token_type = json_request['token_type']
        token_expires = json_request['expires_in']
        print('Acesso autorizado e dados extraídos!')
    else:
        print('Acesso não autorizado!')
        
    return access_token, token_type, token_expires

In [434]:
access_token, token_type, token_expires = request_auth()

Acesso autorizado e dados extraídos!


('BQArRQj-h96XT6diwbZ41YPfRhEx1H8qsw0DPgHsfcViHD3tliXfyj72R8a0LCphUurP-j_lgFwLk8T2s_w0sRXIYjL6GqMiYliVT3NlHC6YFU3b6Es',
 'Bearer',
 3600)

In [298]:
def get_profile(token_type, access_token):
    headers = {
        'Authorization': token_type + ' ' + access_token
    }
    url = 'https://api.spotify.com/v1/me/'

    response = get(url=url, headers=headers)
    
    return response

In [300]:
profile = get_profile(token_type, access_token)
profile.json()

{'error': {'status': 401, 'message': 'Valid user authentication required'}}

In [302]:
good_songs = 'https://open.spotify.com/playlist/6DI0NiX9bE3fIF6cEoI2zL?si=ef8a610d53f64627'
bad_songs = 'https://open.spotify.com/playlist/6IBody2iNg5TgmAeYiHYpW?si=xsEvNjbbQYqt0rs9wP3yOg'

In [366]:
def get_playlist(playlist, token_type, access_token):
    playlist_id = playlist.split('/')[-1].split('?')[0]
    endpoint = f'https://api.spotify.com/v1/playlists/{playlist_id}/tracks'
    headers={
        'Authorization': token_type + ' ' + access_token
    }

    response = get(url=endpoint, headers=headers)

    return response.json()

In [374]:
playlist = get_playlist(good_songs, token_type, access_token)

In [444]:
len(playlist['items'])

100

In [479]:
playlist['items'][0]['track']['album'][['id', 'name']]

TypeError: unhashable type: 'list'

In [541]:
def json_to_dataframe(playlist, label=None):
    m = len(playlist['items'])
    artist = list()
    duration_ms = list()
    href = list()
    id = list()
    name = list()
    popularity = list()
    
    for i in range(m):
        artist.append(playlist['items'][i]['track']['artists'])
        duration_ms.append(playlist['items'][i]['track']['duration_ms'])
        href.append(playlist['items'][i]['track']['href'])
        id.append(playlist['items'][i]['track']['id'])
        name.append(playlist['items'][i]['track']['name'])
        popularity.append(playlist['items'][i]['track']['popularity'])
    
    if label == 1:
        y = np.zeros((m, 1))
    elif label == 0:
        y = np.ones((m, 1))

    features = np.asarray([artist, duration_ms, href, id, name, popularity], dtype=np.float32)
    
    dataframe = pd.concat(features, axis=0)
    
    return features

In [543]:
df = json_to_dataframe(playlist, label=1)
df[-3]

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (6, 100) + inhomogeneous part.